# **Bitcoin Cycle Extremes Forecast: Tops and Bottoms**

A supervised classifier that answers *when*, not *how high*: for every day it estimates the
probability that a cycle **top** (or **bottom**) falls within the next 1 month, 6 months or 1 year.

- **Labels**: the run-up window before each historical extreme. Tops
  2011-06-08, 2013-11-30, 2017-12-17, 2021-11-10, 2025-10-06 - bottoms
  2011-11-19, 2015-01-14, 2018-12-15, 2022-11-21, 2026-07-01.
- **Features**: price technicals, halving-epoch phase, market data (Yahoo), macro (FRED, moved to
  its release date), days since the last top/bottom and the Cycle clock. CoinMetrics on-chain
  fields are left out: its community file lags by months, so they would sit frozen through the
  latest part of the test window.
- **Split**: nothing from `split_date` on is trained on, and nothing fitted to the extremes looks
  past it either - the cycle clock is refitted on the extremes before `split_date`, and an extreme
  only enters "days since last top/bottom" 90 days after it happened. That keeps the 2025 top and
  the 2026 bottom a genuine out-of-sample test: `evaluate_out_of_sample` scores them (AUC + how
  many days the implied date misses the real one).
- **ThermoModel** features are off (`use_thermomodel=False`): `data/thermomodel.csv` is fitted on
  90% of the history, which reaches into the test window.
- **Output**: `output/0.Forecast_Tops.jpg` (tops) and
  `output/0.Forecast_Bottoms.jpg` (bottoms), saved by the pipeline itself. A forecast line
  is only drawn while it is still open: a signal already answered by a labelled extreme is not
  shown as the prediction of a further one.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys, os
_base = os.getcwd()
while _base and not os.path.isdir(os.path.join(_base, 'functions')):
    _parent = os.path.dirname(_base)
    if _parent == _base: break
    _base = _parent
if os.path.abspath(_base) not in sys.path:
    sys.path.insert(0, os.path.abspath(_base))
from functions.functions_Forecast import run_bitcoin_forecast

PARAMS = dict(
    load_markets=True,
    load_fred=True,
    use_thermomodel=False,  # leaks the test window, see above
    use_cycles=True,
    data_folder=os.path.join(_base, "data"),
    activation_threshold=0.9,
    split_date="2024-01-01",
)

In [ ]:
tops = run_bitcoin_forecast(side="top", **PARAMS)

In [ ]:
bottoms = run_bitcoin_forecast(side="bottom", **PARAMS)